# Day 037 — Exercise 2: coerce_numeric_columns

**What you'll build:** `coerce_numeric_columns(df, columns) -> pd.DataFrame` — convert listed columns to numeric using `pd.to_numeric(errors='coerce')`, turning non-numeric strings into NaN.

**Why it matters:** Real CSVs often store numbers as strings, or have typos like `'abc'` in a price field. `errors='coerce'` converts what it can and makes bad values NaN — a safe, auditable conversion that you can then fill with your null strategy.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import io
import pandas as pd

import pandas as pd

def drop_or_fill_nulls(df: pd.DataFrame, strategy: str = 'mean') -> pd.DataFrame:
    result   = df.copy()
    if strategy == 'drop':
        return result.dropna().reset_index(drop=True)
    num_cols = result.select_dtypes(include='number').columns
    if strategy == 'zero':
        result[num_cols] = result[num_cols].fillna(0)
    elif strategy == 'mean':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].mean())
    elif strategy == 'median':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].median())
    else:
        raise ValueError(
            f"Unknown strategy {strategy!r}. "
            "Use 'drop', 'zero', 'mean', or 'median'."
        )
    return result


# COERCE_DF: 'abc' and 'def' are bad values that will become NaN
COERCE_CSV = (
    'item,price,quantity\n'
    'pen,5,10\n'
    'book,abc,5\n'
    'laptop,999,def\n'
    'cup,12.5,20'
)
COERCE_DF = pd.read_csv(io.StringIO(COERCE_CSV))
# price dtype is 'object' because 'abc' prevented numeric inference

## Your Implementation

In [ ]:
def coerce_numeric_columns(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Convert the listed columns to numeric dtype.

    Values that cannot be converted become NaN (errors='coerce').
    Original df is not mutated.
    """
    result = df.copy()
    # TODO: for col in columns:
    #     result[col] = pd.to_numeric(result[col], errors='coerce')
    # TODO: return result
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined, returns DataFrame
    try:
        assert 'coerce_numeric_columns' in globals()
        result = coerce_numeric_columns(COERCE_DF, ['price'])
        assert isinstance(result, pd.DataFrame), \
            f'expected DataFrame, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: returns a DataFrame')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: non-numeric 'abc' becomes NaN in price
    try:
        result = coerce_numeric_columns(COERCE_DF, ['price', 'quantity'])
        import math
        assert math.isnan(float(result.at[1, 'price'])), \
            f"price[1] should be NaN (was 'abc'), got {result.at[1, 'price']}"
        passed += 1; print("\u2705 Check 2: 'abc' becomes NaN in price column")
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: valid numbers are converted to float
    try:
        result = coerce_numeric_columns(COERCE_DF, ['price', 'quantity'])
        assert float(result.at[0, 'price']) == 5.0, \
            f"price[0] should be 5.0, got {result.at[0, 'price']}"
        assert float(result.at[2, 'price']) == 999.0, \
            f"price[2] should be 999.0, got {result.at[2, 'price']}"
        passed += 1; print('\u2705 Check 3: valid numbers converted to float correctly')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: coerced columns are now numeric dtype
    try:
        result = coerce_numeric_columns(COERCE_DF, ['price', 'quantity'])
        assert pd.api.types.is_numeric_dtype(result['price']), \
            f"price dtype still {result['price'].dtype}"
        assert pd.api.types.is_numeric_dtype(result['quantity']), \
            f"quantity dtype still {result['quantity'].dtype}"
        passed += 1; print('\u2705 Check 4: coerced columns are numeric dtype')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: original DataFrame not mutated
    try:
        _ = coerce_numeric_columns(COERCE_DF, ['price', 'quantity'])
        assert COERCE_DF['price'].dtype == object, \
            f'original price dtype changed to {COERCE_DF["price"].dtype}'
        assert COERCE_DF.at[1, 'price'] == 'abc', \
            f"original price[1] changed from 'abc' to {COERCE_DF.at[1, 'price']}"
        passed += 1; print('\u2705 Check 5: original DataFrame not mutated')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import pandas as pd

def coerce_numeric_columns(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    result = df.copy()
    for col in columns:
        result[col] = pd.to_numeric(result[col], errors='coerce')
    return result
```

</details>